# Ch 9　Durable execution 與 fault tolerance

> **本 notebook 不需要 API key。** 我們讓一個節點「前兩次故意失敗、第三次成功」，親眼看 RetryPolicy 把它救回來。

In [ ]:
!uv pip install -q langgraph

## 9.5　Retry：暫時性故障自動重試

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy

class State(TypedDict):
    result: str

# 用一個會「越試越好」的假 API 模擬暫時性故障（像網路抖動、5xx）。
class TransientError(Exception):   # 不是 ValueError/TypeError 那類 bug，預設會被重試
    pass

attempts = {"n": 0}
def flaky_node(state: State):
    attempts["n"] += 1
    print(f"  第 {attempts['n']} 次嘗試…")
    if attempts["n"] < 3:
        raise TransientError("API 暫時抽風 😵")   # 前兩次故意炸
    return {"result": f"第 {attempts['n']} 次終於成功！"}

builder = StateGraph(State)
# max_attempts=3：最多試 3 次（含第一次）。預設 backoff 會在重試間等待，這裡縮短方便示範。
builder.add_node("flaky", flaky_node,
                 retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01, backoff_factor=1.0))
builder.add_edge(START, "flaky"); builder.add_edge("flaky", END)
graph = builder.compile()

print("結果:", graph.invoke({"result": ""}))   # 前兩次失敗會被自動重試，第三次成功

## 哪些例外「不會」被重試？

預設 `default_retry_on` 把 `ValueError`、`TypeError`、`RuntimeError` 等視為「你的程式 bug」，**不重試**（重試也只是再錯一次）。retry 是給暫時性故障用的，不是給 bug。

In [ ]:
attempts2 = {"n": 0}
def buggy_node(state: State):
    attempts2["n"] += 1
    print(f"  第 {attempts2['n']} 次嘗試…")
    raise ValueError("這是程式 bug，不是暫時故障")   # ValueError 預設不重試

b2 = StateGraph(State)
b2.add_node("buggy", buggy_node, retry_policy=RetryPolicy(max_attempts=3))
b2.add_edge(START, "buggy"); b2.add_edge("buggy", END)
g2 = b2.compile()

try:
    g2.invoke({"result": ""})
except ValueError as e:
    print("如預期：只試了一次就冒出來 ->", e)   # 只會看到「第 1 次嘗試」

## 9.4　durability mode：效能 vs 安全的旋鈕

`durability="sync"`（每步寫完才繼續，最安全）/ `"async"`（邊跑邊寫）/ `"exit"`（只在結束時寫，最快）。這裡示範它能正常執行；選哪個依你的回復需求。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
g3 = builder.compile(checkpointer=InMemorySaver())   # 注意：durability 需要 checkpointer
attempts["n"] = 0   # 重設計數器
for part in g3.stream({"result": ""}, {"configurable": {"thread_id": "d1"}}, durability="sync"):
    print("step:", part)

## 重點回顧

只要有 checkpointer 就有 durable execution；但 resume 會 **replay**，所以副作用要包進 task、要冪等。節點韌性靠 RetryPolicy（暫時故障）+ timeout + error handler，順序是「重試 → 用盡才 error handler」。